# 👑 Notebook 3: Leader Election with Auto-Renewing Leases

A lease plus a background renew loop is all you need for leader election: one process holds
the lease and keeps it alive; if it stops renewing, the lease expires on its own and a standby
takes over.

This is exactly how **Kubernetes leader election**, **etcd sessions**, and **ZooKeeper
ephemeral nodes** work under the hood.

We'll simulate:
1. Three worker processes competing for leadership.
2. The leader auto-renews via a background thread.
3. The leader "crashes" halfway through.
4. A standby detects the expiry and takes over.
5. A matplotlib timeline shows who was leader, when.
6. A zombie leader wakes up and tries to write — and gets fenced.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/lease
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload
the window: `Cmd+Shift+P` → **Reload Window**.


## 1. The lease store

In a real cluster this lives in etcd/ZooKeeper/Redis. Here it's a plain object with a
`threading.Lock` so our concurrent workers don't step on each other while reading/writing it.
**The Lease logic itself is what you'd implement server-side.**


In [ ]:
import time, threading
from dataclasses import dataclass
from typing import Optional

@dataclass
class Lease:
    holder: Optional[str] = None
    expires_at: float = 0.0
    # fencing token: monotonically increases every time a NEW holder acquires.
    # Downstream services can reject stale writes from a previously-expired leader.
    fencing_token: int = 0

    _mu: threading.Lock = None  # type: ignore

    def __post_init__(self):
        self._mu = threading.Lock()

    def _expired_locked(self) -> bool:
        return time.monotonic() >= self.expires_at

    def acquire(self, who: str, ttl: float):
        with self._mu:
            if self.holder is None or self._expired_locked():
                self.holder = who
                self.expires_at = time.monotonic() + ttl
                self.fencing_token += 1
                return True, self.fencing_token
            return False, None

    def renew(self, who: str, ttl: float) -> bool:
        with self._mu:
            if self.holder == who and not self._expired_locked():
                self.expires_at = time.monotonic() + ttl
                return True
            return False

    def snapshot(self):
        with self._mu:
            return self.holder, self.expires_at, self.fencing_token

print('Lease store ready.')


## 2. The worker

Each worker runs a loop:

1. Try to `acquire` the lease.
2. If it wins, start renewing in the background at `TTL / 3` — the rule of thumb is to renew
   **three times more often** than the TTL, so one dropped renew doesn't cost you leadership.
3. If it loses, sleep and retry.

We also record every state change into a shared `events` list so we can plot the timeline at
the end.


In [ ]:
events = []           # (timestamp, worker, state) tuples
events_lock = threading.Lock()
T0 = time.monotonic()

def log(worker, state):
    with events_lock:
        events.append((time.monotonic() - T0, worker, state))

class Worker(threading.Thread):
    def __init__(self, name, lease, ttl=1.0, crash_after=None, stop_event=None):
        super().__init__(daemon=True)
        self.name = name
        self.lease = lease
        self.ttl = ttl
        self.crash_after = crash_after   # seconds of leadership before we 'crash'
        self.stop_event = stop_event
        self.token = None

    def run(self):
        log(self.name, 'start')
        became_leader_at = None
        while not self.stop_event.is_set():
            # Am I already leader? If yes, renew.
            if self.token is not None:
                ok = self.lease.renew(self.name, self.ttl)
                if not ok:
                    log(self.name, 'lost')
                    self.token = None
                    became_leader_at = None
                    continue
                # Simulate crash: stop renewing and exit the thread.
                if self.crash_after and (time.monotonic() - became_leader_at) >= self.crash_after:
                    log(self.name, 'crash')
                    return
                time.sleep(self.ttl / 3)
                continue

            # Not  try to acquire.leader 
            ok, token = self.lease.acquire(self.name, self.ttl)
            if ok:
                self.token = token
                became_leader_at = time.monotonic()
                log(self.name, f'leader(token={token})')
            else:
                time.sleep(self.ttl / 5)
        log(self.name, 'stop')

print('Worker class ready.')


## 3. Run the election

- Three workers: A, B, C.
- A is configured to "crash" ~1.5s after becoming leader.
- Total simulation: 5 seconds.


In [ ]:
lease = Lease()
stop = threading.Event()

workers = [
    Worker('A', lease, ttl=1.0, crash_after=1.5, stop_event=stop),
    Worker('B', lease, ttl=1.0, stop_event=stop),
    Worker('C', lease, ttl=1.0, stop_event=stop),
]
for w in workers:
    w.start()

time.sleep(5.0)
stop.set()
for w in workers:
    w.join(timeout=1.0)

holder, _, token = lease.snapshot()
print(f'Final holder: {holder}  fencing_token={token}')
print(f'Captured {len(events)} events.')

leader_events = [(t, who) for t, who, st in events if st.startswith('leader')]
# 1. A led first, then somebody else took over after the crash.
assert leader_events[0][1] == 'A', leader_events
assert len(leader_events) >= 2, 'nobody took over after A crashed'
assert leader_events[1][1] != 'A', leader_events
# 2. Tokens are strictly increasing — that is what makes them usable for fencing.
assert token >= 2, f'expected at least two grants, got token={token}'
# 3. Failover was bounded by the TTL, not open-ended.
crash_t = next(t for t, who, st in events if st == 'crash')
takeover_t = leader_events[1][0]
assert 0 < takeover_t - crash_t < 2 * 1.0, f'failover took {takeover_t - crash_t:.2f}s for a 1.0s TTL'
print(f'✔ failover took {takeover_t - crash_t:.2f}s (TTL = 1.0s), new token = {token}')
for t, who, st in events:
    print(f'  t={t:5.2f}s  {who}  {st}')


## 4. Timeline

The gap between the bars is the failover window. It is bounded by the TTL: shorter TTL means
faster failover, but more renew traffic and a higher chance a GC pause costs you the lease.


In [ ]:
import matplotlib.pyplot as plt

# Derive leadership intervals from events.
names = ['A', 'B', 'C']
intervals = {n: [] for n in names}
active = {n: None for n in names}

for t, who, st in events:
    if st.startswith('leader'):
        active[who] = t
    elif st in ('crash', 'lost', 'stop') and active[who] is not None:
        intervals[who].append((active[who], t))
        active[who] = None

# Close any still-open interval at the end of the sim.
end_t = events[-1][0] if events else 0
for n in names:
    if active[n] is not None:
        intervals[n].append((active[n], end_t))

fig, ax = plt.subplots(figsize=(9, 2.5))
colors = {'A': '#e06666', 'B': '#6aa84f', 'C': '#3c78d8'}
for i, n in enumerate(names):
    for (s, e) in intervals[n]:
        ax.barh(i, e - s, left=s, color=colors[n], edgecolor='black')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('time (s)')
ax.set_title('Lease holder over time (gap = TTL-bounded failover window)')
ax.set_xlim(0, end_t + 0.2)
plt.tight_layout()
plt.show()


## 5. What the timeline shows

1. **A acquired first** (fencing token = 1) and started its renew loop.
2. After ~1.5s, A **stopped renewing** (simulated crash).
3. For up to `TTL` seconds nobody could take over — the lease was still "valid" from the
   system's point of view. **This is the price of not having a human say "A is dead".**
4. As soon as `expires_at` passed, B (or C) successfully `acquire`d with a **new fencing
   token = 2**.

### ⚠️ The lease alone is not safety

A zombie leader — A unpaused after a long GC pause — still *thinks* it is the leader and will
happily send writes to a database. The lease expiring on the server changes nothing about what
A believes. The database must reject any write whose fencing token is **less than the highest
token it has seen**. We demonstrate that in §6 below; the `split-brain-and-fencing` lab goes
deeper.

### TTL trade-off

| TTL | Pros | Cons |
|---|---|---|
| Short (e.g. 1s) | fast failover | lots of renew traffic; spurious expirations on GC/network hiccups |
| Long (e.g. 30s) | cheap; tolerant of hiccups | slow failover; more dead time |

Rule of thumb in production (etcd, k8s):
- renew `interval ≈ TTL / 3`
- `TTL >> (max GC pause + max network RTT + clock skew budget)`

### Where you meet this in real systems

- **Kubernetes controller-manager / scheduler** — leader election via a `Lease` resource in the
  API server (`kube-system/kube-scheduler`). Default lease duration 15s, renewed every 10s.
- **etcd** — `Lease` API with a TTL plus a `KeepAlive` stream (auto-renewal from the client library).
- **ZooKeeper** — ephemeral nodes tied to the client session TTL.
- **Consul** — sessions with a TTL; services attach locks to sessions.
- **Redis** — `SET key val NX PX <ttl>` plus periodic `SET ... XX PX <ttl>` (see Redlock, and
  Kleppmann's critique of it — the missing piece is exactly the fencing token).


## 6. 🧟 The zombie leader — run it, don't just warn about it

A "crashed" leader in this simulation simply stopped renewing. A *real* GC pause does something
worse: the process comes back, its `self.token` is still set, and it resumes exactly where it
left off. Below we replay that with the tokens the election above actually produced.

In [ ]:
class FencedDB:
    """The resource. It never asks who the leader is — it only compares tokens."""

    def __init__(self):
        self.highest_token = 0
        self.rows: list[tuple[str, int, str]] = []

    def write(self, who: str, token: int, value: str) -> bool:
        if token < self.highest_token:
            print(f'  ❌ FENCED {who}@{token}: highest seen is {self.highest_token}')
            return False
        self.highest_token = token
        self.rows.append((who, token, value))
        print(f'  ✅ {who}@{token} wrote {value!r}')
        return True


db = FencedDB()
zombie_token = 1                      # what A was holding before it paused
current_token = lease.snapshot()[2]   # what the live leader holds now
current_holder = lease.snapshot()[0]
assert current_token > zombie_token

print('the live leader does its work:')
assert db.write(current_holder, current_token, 'checkpoint-42')
assert db.write(current_holder, current_token, 'checkpoint-43'), \
    'the current leader must be able to write more than once per term'

print('\nA unpauses and resumes what it was doing:')
assert db.write('A', zombie_token, 'checkpoint-OLD') is False

print('\nrows in the database:')
for row in db.rows:
    print(' ', row)

# The invariant: nothing from an expired lease ever reached the resource.
assert all(tok == current_token for _, tok, _ in db.rows), db.rows
assert not any('OLD' in v for _, _, v in db.rows)
print(f'\n✔ the zombie held a valid-looking lease object and still could not write.')
print('  The lease bounded how long we WAIT; the token is what makes it SAFE.')